# Auditoria da base bruta: Licitações Federais 2025

Levantamento dos problemas dos dados antes da limpeza.
Neste notebook os dados são apenas **observados e contados**. Nada é alterado aqui.
Os números encontrados vão para o README, na seção que descreve as inconsistências da base.

Arquivos analisados (pasta `data/raw/`):
- `compras.csv`: cada linha é uma compra do governo (o edital)
- `itens.csv`: cada linha é um item dentro de uma compra
- `resultados.csv`: cada linha é um fornecedor que participou da disputa de um item

## 1. Abertura dos arquivos

Abrir os três arquivos lendo tudo como texto.
Assim é possível ver os dados exatamente como chegaram, sem que o pandas altere nada automaticamente.

In [1]:
import pandas as pd

# Abrir os três arquivos lendo tudo como texto (dtype=str)
# e sem transformar "N/A" ou "NULL" em vazio (keep_default_na=False)
compras = pd.read_csv("../data/raw/compras.csv", dtype=str, keep_default_na=False)
itens = pd.read_csv("../data/raw/itens.csv", dtype=str, keep_default_na=False)
resultados = pd.read_csv("../data/raw/resultados.csv", dtype=str, keep_default_na=False)

# Mostrar o tamanho de cada tabela: (linhas, colunas)
print("compras:", compras.shape)
print("itens:", itens.shape)
print("resultados:", resultados.shape)


compras: (2042, 25)
itens: (7077, 14)
resultados: (15896, 13)


**Resultado:** os três arquivos chegaram completos. As quantidades de linhas (2.042 compras, 7.077 itens e 15.896 resultados) batem com a prova real.

## 2. Conhecendo as colunas

Para cada coluna: ver um exemplo de valor, contar quantos valores diferentes existem e contar quantas células estão vazias.
O significado de cada coluna está no anexo 7 do enunciado; aqui é conferido o que realmente chegou.

In [2]:
def perfil(tabela):
    """Recebe uma tabela e devolve um resumo de cada coluna:
    um valor de exemplo (o da primeira linha), quantos valores diferentes existem
    e quantas células estão vazias."""
    return pd.DataFrame({
        'exemplo': tabela.iloc[0],
        'distintos': tabela.nunique(),
        'vazios': (tabela == '').sum()
    })
perfil(compras)

,exemplo,distintos,vazios
numero_controle_pncp,00889834000108-1-000053/2023,2042,0
ano_compra,2023,5,0
sequencial_compra,53,1140,0
numero_compra,90037,570,0
processo,0,1886,0
objeto_compra,Contratação de serviços continuados de comunic...,1980,0
srp,False,2,0
cnpj_orgao,00889834000108,539,0
razao_social_orgao,FUND COORD DE APERFEICOAMENTO DE PESSOAL DE NI...,440,0
poder_id,E,3,0


**O que a tabela `compras` mostra:**
- Cada compra aparece uma vez só: `numero_controle_pncp` tem 2.042 valores diferentes em 2.042 linhas.
- `cnpj_orgao` tem 539 valores diferentes, mas o enunciado espera 363 órgãos compradores (anexo 8). Sinal de que o mesmo CNPJ aparece escrito de jeitos diferentes (ver seção 4).
- `razao_social_orgao` tem 440 nomes diferentes: o mesmo órgão escrito de várias formas.
- Só na primeira linha já aparecem formatos diferentes: dinheiro como `20307804.45` e como `19.290.383,46`, data como `2025-01-06T11:51:06` e como `06.03.2025`.
- `processo` traz o valor `0`, que é um "nulo disfarçado" (um texto que significa "sem informação").
- `uf_sigla` (27) e `modalidade_id` (7) estão corretas: 26 estados mais o DF, e as 7 modalidades do enunciado.
- As únicas células vazias de verdade estão nas datas de abertura e de encerramento das propostas (617 cada), colunas que não entram no modelo.

In [3]:
perfil(itens)

,exemplo,distintos,vazios
numero_controle_pncp,00889834000108-1-000053/2023,2042,0
numero_item,1,43,0
descricao,Resenha de Notícia / Agenciamento de Informaçã...,3633,0
material_ou_servico,Serviço,2,0
categoria_item,Não se aplica,1,0
quantidade,3.0,1115,0
unidade_medida,UNIDADE,360,9
valor_unitario_estimado,6769268.15,5632,0
valor_total_estimado,"20.307.804,45",6123,0
sigiloso,False,2,0


**O que a tabela `itens` mostra:**
- `numero_controle_pncp` tem 2.042 valores diferentes, a mesma quantidade de compras de `compras.csv`.
- `categoria_item` tem um único valor ("Não se aplica"), como o enunciado avisa no anexo 7. Por isso, a classificação do gasto vai usar `material_ou_servico`, que tem 2 valores.
- `quantidade`, `valor_unitario_estimado` e `valor_total_estimado` misturam formatos de número já na primeira linha: `6769268.15` e `20.307.804,45` (ver seção 4).
- `situacao_item` tem 13 valores diferentes para uma informação simples, o que indica nulos disfarçados (ver seção 5).
- `unidade_medida` tem 360 valores diferentes (um deles é o vazio) e 9 células vazias, e `catalogo_codigo_item` está vazio em 7.068 itens. Nenhuma das duas colunas entra no modelo.

In [4]:
perfil(resultados)

,exemplo,distintos,vazios
numero_controle_pncp,00889834000108-1-000053/2023,2041,0
numero_item,1,41,0
sequencial_resultado,1,1773,0
ordem_classificacao_srp,1.0,3,11976
ni_fornecedor,51.593.855/0001-04,11418,0
tipo_pessoa,PJ,3,0
nome_fornecedor,CRIATIVA DIGITAL COMUNICACOES LTDA,10707,0
porte,Demais,13,0
quantidade_homologada,"R$ 3,00",2203,0
valor_unitario_homologado,"6.430.127,82",6074,0


**O que a tabela `resultados` mostra:**
- `numero_controle_pncp` tem 2.041 compras diferentes, uma a menos que `compras.csv`: pelo menos uma compra não tem nenhum fornecedor classificado.
- `quantidade_homologada` vem escrita como dinheiro (`R$ 3,00`), e `data_resultado` vem em formato brasileiro com hora (`10/03/2026 00:00`), inclusive com data em 2026.
- `ordem_classificacao_srp` está vazia em 11.976 linhas (pela regra 2 do enunciado, vazio vale 1, ou seja, o vencedor).
- `data_cancelamento` está vazia em 15.834 linhas; as preenchidas são os resultados cancelados, contados na seção 3.
- `porte` tem 13 valores diferentes para uma informação que deveria ter poucas categorias: os valores aparecem na seção 5 (nulos disfarçados).
- `ni_fornecedor` traz documento com máscara (`51.593.855/0001-04`).

Observação: o vazio (`""`) também conta como um valor diferente no `nunique()`. Nas colunas que têm vazios, o número de valores reais é o de "distintos" menos 1.

In [5]:
def contar_so_espacos(tabela):
    """Recebe uma tabela e devolve quantas células dela têm só espaços."""
    total = 0                                                
    for coluna in tabela.columns:                               
        total = total + tabela[coluna].str.isspace().sum()   
    return total                                               

print("compras:", contar_so_espacos(compras))
print("itens:", contar_so_espacos(itens))
print("resultados:", contar_so_espacos(resultados))

compras: 0
itens: 0
resultados: 0


**Verificação dos vazios:**
Nos perfis acima, os vazios foram contados como células exatamente iguais a `""`, então uma célula preenchida só com espaços não entraria na conta.
A função `contar_so_espacos` verificou esse caso com `.str.isspace()` e encontrou zero células assim nas três tabelas: a contagem de vazios dos perfis está completa.

## 3. Chaves e ligações entre as tabelas

As três tabelas se ligam assim: uma compra tem vários itens, e um item tem vários resultados (um por fornecedor classificado).
A compra é identificada por `numero_controle_pncp`; o item, pelo par `numero_controle_pncp` + `numero_item`.

Perguntas desta seção: as chaves se repetem? Quantos itens têm resultado? Quantos itens têm mais de um fornecedor classificado?

In [6]:
chave_item = ['numero_controle_pncp', 'numero_item']

print('contratacoes repetidas em compras ....', compras['numero_controle_pncp'].duplicated().sum())
print('itens repetidos em itens .............', itens.duplicated(chave_item).sum())

resultados_por_item = resultados.groupby(chave_item).size()
print('itens que aparecem em resultados .....', len(resultados_por_item))
print('itens com 2 ou mais classificados ....', (resultados_por_item > 1).sum())
print('maior numero de classificados num item', resultados_por_item.max())

print('resultados canelados .................', (resultados["data_cancelamento"].str.strip() != "").sum())
print(itens["tem_resultado"].value_counts())

contratacoes repetidas em compras .... 0
itens repetidos em itens ............. 0
itens que aparecem em resultados ..... 6225
itens com 2 ou mais classificados .... 466
maior numero de classificados num item 1773
resultados canelados ................. 62
tem_resultado
True     6225
False     852
Name: count, dtype: int64


**O que as chaves mostram:**
- Nenhuma chave se repete: cada compra e cada item (`numero_controle_pncp` + `numero_item`) aparecem uma única vez. Isso garante que a tabela fato pode ter uma linha por item.
- 6.225 itens aparecem em `resultados`, exatamente a quantidade de itens marcados `tem_resultado = True`. Os 852 itens marcados `False` não têm resultado e ficam fora da fato (regra 3 do enunciado, que confere com a prova real).
- 466 itens têm mais de um fornecedor classificado, e um deles chega a 1.773. Por isso, antes de montar a fato, é preciso escolher um vencedor por item (regra 2); sem isso, o mesmo item apareceria várias vezes e os valores seriam somados repetidas vezes.
- 62 resultados têm data de cancelamento preenchida e são descartados antes da escolha do vencedor.

## 4. Formatos de dinheiro, data e documento

Primeiro, medir o tamanho do problema: quantos valores de cada coluna numérica o pandas não consegue ler como número.
Depois, descobrir quais formatos causam isso, trocando cada sequência de dígitos por `9`. Assim, `R$ 1.234,56` e `R$ 8.765,43` viram o mesmo formato, `R$ 9.9,9`, e dá para contar os formatos em vez de milhares de valores diferentes.

Valores numéricos

Criar uma função que tenta converter uma coluna em número e conta quantos valores falham, e aplicar a função nas colunas de quantidade e valor de itens e resultados. Com ela é possível medir quanto de cada coluna de valor não está em um formato que o pandas entende.

In [7]:
def contar_nao_numericos(tabela, coluna):
    """Recebe uma tabela e o nome de uma coluna e mostra quantos valores
    dessa coluna não viram número direto, e quanto isso representa do total."""
    convertidos = pd.to_numeric(tabela[coluna], errors="coerce")   
    falhas = convertidos.isnull().sum()                            
    total = len(convertidos)                                      
    print(f"{coluna:30} {falhas:>6} de {total:>6} ({falhas / total:.1%})")

contar_nao_numericos(itens, "quantidade")
contar_nao_numericos(itens, "valor_unitario_estimado")
contar_nao_numericos(itens, "valor_total_estimado")
contar_nao_numericos(resultados, "quantidade_homologada")
contar_nao_numericos(resultados, "valor_unitario_homologado")
contar_nao_numericos(resultados, "valor_total_homologado")

quantidade                       1481 de   7077 (20.9%)
valor_unitario_estimado          1602 de   7077 (22.6%)
valor_total_estimado             1662 de   7077 (23.5%)
quantidade_homologada            3424 de  15896 (21.5%)
valor_unitario_homologado        3614 de  15896 (22.7%)
valor_total_homologado           3832 de  15896 (24.1%)


**O que a contagem mostra:** entre 20,9% e 24,1% dos valores de cada coluna numérica não são lidos como número pelo pandas (de 1.481 quantidades em `itens` a 3.832 valores homologados em `resultados`). O problema aparece nas seis colunas, com tamanho parecido: ele vem do sistema de origem e vai precisar de uma função de conversão própria (desafio 2 e regra 12 do enunciado).

Criar uma função que troca cada bloco de dígitos por 9 e conta os formatos mais comuns, e aplicar a função na coluna valor_unitario_estimado. Assim é possível ver como os valores estão escritos, sem se perder nos números em si.

In [8]:
def ver_formatos(tabela, coluna):
    """Recebe uma tabela e o nome de uma coluna e devolve os formatos mais comuns,
    trocando cada bloco de dígitos por 9."""
    formatos = tabela[coluna].str.replace(r"\d+", "9", regex=True) 
    return formatos.value_counts().head(8)                

ver_formatos(itens, "valor_unitario_estimado")

valor_unitario_estimado
9.9          5475
R$ 9,9        386
R$ 9.9,9      201
R$ 9,9        193
9,9           190
 R$ 9,9       184
R$ 9.9,9       95
9,9.9          87
Name: count, dtype: int64

Contar quantos valores de valor_unitario_estimado viram número direto, para conferir se são exatamente os do formato 9.9.

In [9]:
print('Viram número direto: ',(pd.to_numeric(itens['valor_unitario_estimado'], errors='coerce').notnull().sum()))

Viram número direto:  5475


Contar três problemas que aparecem nos formatos: o símbolo R$, o espaço invisível (\xa0) e o espaço no início do texto.

In [10]:
coluna = itens["valor_unitario_estimado"]

print("com R$:               ", coluna.str.contains("R$", regex=False).sum())
print("com espaço invisível: ", coluna.str.contains("\xa0", regex=False).sum())
print("começam com espaço:   ", coluna.str.startswith(" ").sum())


com R$:                1188
com espaço invisível:  301
começam com espaço:    260


Repetir a contagem na coluna quantidade, para ver se os mesmos problemas aparecem em uma coluna que não é de dinheiro.

In [11]:
coluna = itens["quantidade"]

print("com R$:               ", coluna.str.contains("R$", regex=False).sum())
print("com espaço invisível: ", coluna.str.contains("\xa0", regex=False).sum())
print("começam com espaço:   ", coluna.str.startswith(" ").sum())

com R$:                1148
com espaço invisível:  264
começam com espaço:    287


**O que os formatos de número mostram:**
- Em `valor_unitario_estimado`, só o formato `9.9` (5.475 valores) é lido direto pelo pandas. É exatamente a diferença entre o total (7.077) e as falhas da contagem anterior (1.602).
- Os outros formatos misturam: real com vírgula decimal (`R$ 9,9`), real com ponto de milhar (`R$ 9.9,9`), número com vírgula decimal (`9,9`) e o formato americano, com vírgula de milhar (`9,9.9`).
- O formato `R$ 9,9` aparece duas vezes na lista: numa delas, o espaço depois do "R$" é um espaço invisível (`\xa0`). Esse caractere aparece em 301 preços e 264 quantidades.
- Há valores com espaço antes do texto: 260 preços e 287 quantidades começam com espaço (` R$ 9,9`).
- O "R$" aparece até onde não deveria: 1.148 **quantidades** vêm escritas como dinheiro.
- Conclusão para a limpeza: tirar os espaços (inclusive o invisível) e o "R$", e decidir qual símbolo é o decimal (regra 12 do enunciado). É o que fazem as funções `limpar_texto` e `converter_numero` do kit.

Formatos das datas

Ver os formatos das colunas de data, trocando cada bloco de dígitos por 9. Assim é possível ver de quantas formas diferentes a mesma informação aparece escrita.

In [12]:
ver_formatos(compras, "data_publicacao")

data_publicacao
9-9-9T9:9:9    1319
9/9/9           233
9-9-9           138
9/9/9 9:9       126
9.9.9           120
9               106
Name: count, dtype: int64

In [13]:
ver_formatos(resultados, "data_resultado")

data_resultado
9-9-9        11213
9/9/9         1889
9.9.9          937
9/9/9 9:9      934
9              923
Name: count, dtype: int64

Somar as contagens de cada formato para conferir se a lista cobre todas as linhas da tabela.

In [14]:
print(ver_formatos(compras, "data_publicacao").sum())
print(ver_formatos(resultados, "data_resultado").sum())

2042
15896


Separar as datas formadas só por dígitos (formato 9) e ver alguns exemplos, para descobrir o que esse número representa.

In [15]:
so_digitos = compras["data_publicacao"].str.isdigit()
print(so_digitos.sum())
compras.loc[so_digitos, "data_publicacao"].head()

106


13     20250103
37     20250108
71     20250115
95     20250117
112    20250122
Name: data_publicacao, dtype: str

Conferir se todas as datas formadas só por dígitos têm o mesmo tamanho.

In [16]:
colados = compras.loc[so_digitos, "data_publicacao"]
colados.str.len().value_counts()

data_publicacao
8    106
Name: count, dtype: int64

Separar a primeira parte das datas escritas com barra (antes da primeira /). Se algum valor passar de 12, essa parte não pode ser o mês, e o padrão é dia/mês/ano.

In [17]:
tem_barra = compras["data_publicacao"].str.contains("/", regex=False)
primeira_parte = compras.loc[tem_barra, "data_publicacao"].str.split("/").str[0]
primeira_parte.head()

2     02
12    02
15    03
33    08
34    08
Name: data_publicacao, dtype: object

Converter a primeira parte em número inteiro e ver o maior e o menor valor.

In [18]:
numeros = primeira_parte.astype(int)
print("maior:", numeros.max())
print("menor:", numeros.min())

maior: 2025
menor: 1


O maior valor foi 2025: parte das datas com barra começa pelo ano, e outra parte começa com dois dígitos. A máscara por bloco (9/9/9) juntava os dois padrões. Por isso, a próxima análise troca cada dígito por 9.

Criar uma segunda versão da função, que troca cada dígito (e não cada bloco) por 9. Assim aparece o tamanho de cada parte da data, e fica possível separar ano/mês/dia de dia/mês/ano.

In [19]:
def ver_formatos_digito(tabela, coluna):
    """Recebe uma tabela e o nome de uma coluna e devolve os formatos mais comuns,
    trocando cada dígito por 9."""
    formatos = tabela[coluna].str.replace(r"\d", "9", regex=True) 
    return formatos.value_counts().head(10)                

ver_formatos_digito(compras, "data_publicacao")

data_publicacao
9999-99-99T99:99:99    1319
99-99-9999              138
99/99/9999 99:99        126
99.99.9999              120
99/99/9999              117
9999/99/99              116
99999999                106
Name: count, dtype: int64

Separar as datas com o ano no fim (começam com dois dígitos e um separador) e ver o maior valor da primeira e da segunda parte. A parte que passar de 12 é o dia.

In [20]:
ano_no_fim = compras["data_publicacao"].str.match(r"\d\d\D")
datas = compras.loc[ano_no_fim, "data_publicacao"]
print("quantas:", ano_no_fim.sum())
print("maior da primeira parte:", datas.str[0:2].astype(int).max())
print("maior da segunda parte: ", datas.str[3:5].astype(int).max())

quantas: 501
maior da primeira parte: 31
maior da segunda parte:  12


Separar as datas com o ano no começo e barra (formato 9999/99/99) e ver o maior valor da segunda e da terceira parte. A parte que passar de 12 é o dia.

In [21]:
ano_no_comeco = compras["data_publicacao"].str.match(r"\d\d\d\d/")
datas = compras.loc[ano_no_comeco, "data_publicacao"]
print("quantas:", ano_no_comeco.sum())
print("maior da segunda parte:  ", datas.str[5:7].astype(int).max())
print("maior da terceira parte: ", datas.str[8:10].astype(int).max())

quantas: 116
maior da segunda parte:   12
maior da terceira parte:  31


Separar as datas com o ano no começo e traço (formato 9999-99-99T99:99:99) e ver o maior valor da segunda e da terceira parte.

In [22]:
ano_com_traco = compras["data_publicacao"].str.match(r"\d\d\d\d-")
datas = compras.loc[ano_com_traco, "data_publicacao"]
print("quantas:", ano_com_traco.sum())
print("maior da segunda parte:  ", datas.str[5:7].astype(int).max())
print("maior da terceira parte: ", datas.str[8:10].astype(int).max())

quantas: 1319
maior da segunda parte:   12
maior da terceira parte:  31


Ver o maior valor da segunda e da terceira parte das datas coladas (8 dígitos), para provar que o padrão é ano, mês e dia.

In [23]:
print("maior da segunda parte: ", colados.str[4:6].astype(int).max())
print("maior da terceira parte:", colados.str[6:8].astype(int).max())

maior da segunda parte:  12
maior da terceira parte: 31


Somar as datas dos quatro grupos para conferir se todas as linhas de compras foram verificadas.

In [24]:
total = ano_no_fim.sum() + ano_no_comeco.sum() + ano_com_traco.sum() + so_digitos.sum()
print("datas verificadas:", total, "de", len(compras))

datas verificadas: 2042 de 2042


Repetir as verificações de padrão na coluna resultados.data_resultado.

In [25]:
ver_formatos_digito(resultados, "data_resultado")

data_resultado
9999-99-99          10301
9999/99/99            960
99.99.9999            937
99/99/9999 99:99      934
99/99/9999            929
99999999              923
99-99-9999            912
Name: count, dtype: int64

In [26]:
ano_no_fim = resultados["data_resultado"].str.match(r"\d\d\D")
datas = resultados.loc[ano_no_fim, "data_resultado"]
print("quantas:", ano_no_fim.sum())
print("maior da primeira parte:", datas.str[0:2].astype(int).max())
print("maior da segunda parte: ", datas.str[3:5].astype(int).max())

quantas: 3712
maior da primeira parte: 31
maior da segunda parte:  12


In [27]:
ano_no_comeco = resultados["data_resultado"].str.match(r"\d\d\d\d/")
datas = resultados.loc[ano_no_comeco, "data_resultado"]
print("quantas:", ano_no_comeco.sum())
print("maior da segunda parte:  ", datas.str[5:7].astype(int).max())
print("maior da terceira parte: ", datas.str[8:10].astype(int).max())

quantas: 960
maior da segunda parte:   12
maior da terceira parte:  31


In [28]:
ano_com_traco = resultados["data_resultado"].str.match(r"\d\d\d\d-")
datas = resultados.loc[ano_com_traco, "data_resultado"]
print("quantas:", ano_com_traco.sum())
print("maior da segunda parte:  ", datas.str[5:7].astype(int).max())
print("maior da terceira parte: ", datas.str[8:10].astype(int).max())

quantas: 10301
maior da segunda parte:   12
maior da terceira parte:  31


In [29]:
so_digitos = resultados["data_resultado"].str.isdigit()
colados = resultados.loc[so_digitos, "data_resultado"]
print("quantas:", so_digitos.sum())
print("maior da segunda parte: ", colados.str[4:6].astype(int).max())
print("maior da terceira parte:", colados.str[6:8].astype(int).max())

quantas: 923
maior da segunda parte:  12
maior da terceira parte: 31


Somar as datas dos quatro grupos para conferir se todas as linhas de resultados foram verificadas.

In [30]:
total = ano_no_fim.sum() + ano_no_comeco.sum() + ano_com_traco.sum() + so_digitos.sum()
print("datas verificadas:", total, "de", len(resultados))

datas verificadas: 15896 de 15896


**O que os formatos de data mostram:**

- A máscara por bloco (`ver_formatos`) mostrava 6 formatos em `compras.data_publicacao` e 5 em `resultados.data_resultado`, mas escondia diferenças: `2025/01/08` e `08/01/2025` viravam os dois `9/9/9`. A máscara por dígito (`ver_formatos_digito`) mostra 7 formatos em cada coluna.
- Há dois padrões de ordem: **dia/mês/ano**, quando o ano está no fim (501 datas em `compras` e 3712 em `resultados`), e **ano/mês/dia**, quando o ano está no começo (com barra, com traço ou colado). Em cada grupo, a parte que chega a 31 é o dia e a que não passa de 12 é o mês.
- As datas coladas (106 em `compras` e 923 em `resultados`) têm 8 dígitos, no padrão ano, mês e dia.
- Alguns valores trazem hora junto (`T11:51:06` ou ` 14:30`). Para a data, a hora não importa.
- Os quatro grupos somam todas as linhas das duas tabelas (2042 e 15896), então nenhuma data ficou sem verificação.
- Conclusão para a limpeza: a função `converter_data` do kit corta os 10 primeiros caracteres (a hora sai) e testa os 6 formatos da lista `FORMATOS_DATA`, que são exatamente os padrões encontrados. O maior valor prova a ordem dia/mês, mas não detecta datas impossíveis como `31/02/2025`. Essas ficam para a conversão (`strptime`), que devolve vazio quando a data não existe.

Documentos (CNPJ e CPF)

Ver os formatos dos documentos trocando cada dígito por 9. Um CNPJ completo tem 14 dígitos e um CPF tem 11. Formatos com menos dígitos indicam zeros perdidos no começo.

In [31]:
ver_formatos_digito(compras, "cnpj_orgao")

cnpj_orgao
99999999999999        1658
99.999.999/9999-99     176
 99999999999999        131
999999999999            64
9999999999999           10
99999999999              3
Name: count, dtype: int64

In [32]:
ver_formatos_digito(resultados, "ni_fornecedor")

ni_fornecedor
99999999999999        9216
99999999999           3688
99.999.999/9999-99    1239
 99999999999999        695
999.999.999-99         392
 99999999999           293
9999999999999          175
9999999999             154
999999999999            18
999999999                6
Name: count, dtype: int64

Somar as contagens do top 10 de ni_fornecedor e comparar com o total de linhas, para saber se algum formato ficou de fora.

In [33]:
print(ver_formatos_digito(resultados, "ni_fornecedor").sum(), "de", len(resultados))

15876 de 15896


Contar os valores de tipo_pessoa, que indica se o documento do fornecedor deveria ser um CNPJ ou um CPF.

In [34]:
resultados["tipo_pessoa"].value_counts()

tipo_pessoa
PJ    11344
PF     4531
PE       21
Name: count, dtype: int64

Ver os documentos dos fornecedores estrangeiros (PE), que não têm CNPJ nem CPF.

In [35]:
estrangeiro = resultados["tipo_pessoa"] == "PE"
resultados.loc[estrangeiro, "ni_fornecedor"].tolist()

['ESTRANG0002421',
 'ESTRANG0002421',
 '\xa0ESTRANG0002421 ',
 'ESTRANG0002421',
 'ESTRANG0002421',
 '\xa0ESTRANG0002421 ',
 'ES.TRA.NG0/0024-21',
 'US95543',
 '96175859',
 '679124431',
 'DUNS-97092690581',
 'EX150AGIA',
 'PF1500001',
 'PF1500001',
 'EX2544919',
 'EX2544919',
 'EX1540461',
 'EX1540479',
 'EX0000000',
 'EX0000000',
 '452442841']

Tirar tudo que não é dígito dos documentos e contar o tamanho que sobra, separando pessoa jurídica (CNPJ, 14 dígitos) e pessoa física (CPF, 11 dígitos). Tamanhos menores indicam zeros perdidos no começo.

In [36]:
digitos = resultados["ni_fornecedor"].str.replace(r"\D", "", regex=True)
print(digitos[resultados["tipo_pessoa"] == "PJ"].str.len().value_counts())
print(digitos[resultados["tipo_pessoa"] == "PF"].str.len().value_counts())

ni_fornecedor
14    11150
13      175
12       18
11        1
Name: count, dtype: int64
ni_fornecedor
11    4372
10     154
9        4
8        1
Name: count, dtype: int64


**O que os formatos de documento mostram:**

- `compras.cnpj_orgao` aparece de 6 formas: só dígitos, com máscara (`99.999.999/9999-99`), com espaço no começo, e com 12, 13 ou 11 dígitos (64, 10 e 3 casos), que são CNPJ que perderam zeros à esquerda.
- Em `resultados.ni_fornecedor`, o top 10 de formatos deixa 20 linhas de fora (15876 de 15896). São quase todas de fornecedores estrangeiros (`tipo_pessoa = PE`, 21 linhas), com documentos sem padrão: letras (`ESTRANG...`, `EX...`, `DUNS-...`), números com 8 ou 9 dígitos, e o mesmo documento escrito de jeitos diferentes, inclusive com máscara de CNPJ aplicada por engano (`ES.TRA.NG0/0024-21`).
- Tirando tudo que não é dígito, os PJ (CNPJ, 14 dígitos) têm 175 documentos com 13 dígitos, 18 com 12 e 1 com 11. Os PF (CPF, 11 dígitos) têm 154 com 10 dígitos, 4 com 9 e 1 com 8. Nenhum documento tem dígitos a mais.
- Conclusão para a limpeza: a função `limpar_documento` do kit tira a máscara, usa o `tipo_pessoa` para saber o tamanho certo (14 ou 11) e devolve os zeros perdidos com `.zfill`. Para estrangeiros, ela só limpa espaços e põe em maiúsculas, então `ESTRANG0002421` e `ES.TRA.NG0/0024-21` continuam diferentes. Esse ponto fica anotado no README como melhoria futura.

## 5. Nulos disfarçados

Um campo sem informação nem sempre aparece vazio: às vezes vem escrito `N/A`, `NULL`, `-`, `0` ou "Não informado". Para o pandas, isso é um texto comum, por isso a função `perfil` não contou esses valores como vazios. Aqui o objetivo é encontrar esses textos e medir quantos são.

Contar os valores da coluna `porte`, que deveria ter só três categorias (ME, EPP e Demais), para ver de quantas formas a ausência de informação aparece escrita.

In [37]:
resultados["porte"].value_counts()

porte
ME                7101
Não Informado     4415
Demais            3893
N/A                 72
NULL                64
n/a                 60
SEM INFORMAÇÃO      60
0                   57
NÃO INFORMADO       53
-                   52
Null                51
EPP                  9
Não se aplica        9
Name: count, dtype: int64

Contar quantos valores de porte são uma das três categorias válidas e quantos são ausência de informação disfarçada.

In [39]:
validos = resultados["porte"].isin(["ME", "EPP", "Demais"])
print("válidos:       ", validos.sum())
print("sem informação:", len(resultados) - validos.sum())
print(f"percentual sem informação: {(len(resultados) - validos.sum()) / len(resultados):.1%}")

válidos:        11003
sem informação: 4893
percentual sem informação: 30.8%


Usar a lista de nulos disfarçados do kit (NULOS, em src/limpeza.py) e contar, coluna por coluna, quantos valores são um desses textos. A comparação é feita em maiúsculas e sem espaços nas pontas, como no kit.

In [40]:
NULOS = {"N/A", "NULL", "NONE", "NAN", "-", "0", "SEM INFORMAÇÃO", "SEM INFORMACAO",
         "NÃO INFORMADO", "NAO INFORMADO", "NÃO SE APLICA", "NAO SE APLICA"}

def contar_nulos_disfarcados(tabela):
    """Recebe uma tabela e mostra, coluna por coluna, quantos valores são nulos disfarçados (textos da lista NULOS)."""   
    for coluna in tabela.columns:
        quantos = tabela[coluna].str.strip().str.upper().isin(NULOS).sum()
        if quantos > 0:
            print(f"{coluna:30} {quantos:>6}")

contar_nulos_disfarcados(compras)

processo                           77
modo_disputa                      967


In [41]:
contar_nulos_disfarcados(itens)

categoria_item                   7077
situacao_item                     220
criterio_julgamento              1883


In [42]:
contar_nulos_disfarcados(resultados)

porte                            4893


In [43]:
itens["categoria_item"].value_counts()

categoria_item
Não se aplica    7077
Name: count, dtype: int64

**O que os nulos disfarçados mostram:**

- Em `resultados.porte`, a falta de informação aparece escrita de 10 jeitos diferentes (`Não Informado`, `N/A`, `NULL`, `n/a`, `SEM INFORMAÇÃO`, `0`, `NÃO INFORMADO`, `-`, `Null` e `Não se aplica`). São 4893 de 15896 resultados (30,8%) sem o porte da empresa, contra 11003 com uma das três categorias válidas (ME, EPP e Demais).
- Usando a lista `NULOS` do kit, os nulos disfarçados aparecem em 6 colunas: `processo` (77) e `modo_disputa` (967) em `compras`; `categoria_item` (7077), `situacao_item` (220) e `criterio_julgamento` (1883) em `itens`; e `porte` (4893) em `resultados`.
- Em `categoria_item`, todas as 7077 linhas são nulo disfarçado: a coluna não carrega informação.
- Os casos aparecem só em colunas de texto e de categoria. Nenhuma coluna de número ou de data tem valores da lista.
- Conclusão para a limpeza: a função `limpar_texto` do kit usa a `eh_nulo` para comparar cada valor, em maiúsculas, com a lista `NULOS`, e devolve vazio (`None`) quando encontra. Assim, no banco, a falta de informação fica registrada de um jeito só.

## 6. Acentos quebrados

Quando um texto com acento é gravado em UTF-8 e lido como Latin-1, cada letra acentuada vira dois símbolos: `Aquisição` vira `AquisiÃ§Ã£o`. O sinal é um `Ã` seguido de algo que não é letra maiúscula nem espaço. A condição de "não maiúscula" existe para não confundir com palavras corretas como `NÃO` ou `MÃE`.

Contar, coluna por coluna, quantos valores têm o sinal de acento quebrado.

In [47]:
def contar_acentos_quebrados(tabela):
    """Recebe uma tabela e mostra, coluna por coluna, quantos valores têm o sinal de acento quebrado."""
    for coluna in tabela.columns:
        quantos = tabela[coluna].str.contains(r"Ã[^A-Z ]", regex=True).sum()
        if quantos > 0:
            print(f"{coluna:30} {quantos:>6}")

contar_acentos_quebrados(compras)

objeto_compra                      43
municipio                          28
nome_unidade                       14


In [49]:
contar_acentos_quebrados(itens)

descricao                         116


In [50]:
contar_acentos_quebrados(resultados)

nome_fornecedor                     1


Ver exemplos de municípios com acento quebrado.

In [51]:
quebrado = compras["municipio"].str.contains(r"Ã[^A-Z ]", regex=True)
compras.loc[quebrado, "municipio"].value_counts().head()

municipio
BrasÃ­lia      5
SÃ£o Paulo     4
VitÃ³ria       2
MaceiÃ³        2
SÃ£o Carlos    1
Name: count, dtype: int64

**O que os acentos quebrados mostram:**

- O problema aparece só em colunas de texto: `objeto_compra` (43), `municipio` (28) e `nome_unidade` (14) em `compras`; `descricao` (116) em `itens`; e `nome_fornecedor` (1) em `resultados`.
- Em textos longos (`objeto_compra`, `descricao`), o acento quebrado atrapalha só a leitura. Em `municipio`, `nome_unidade` e `nome_fornecedor`, que viram dimensões, ele é mais grave: `Brasília` e `BrasÃ­lia` virariam dois municípios diferentes.
- Conclusão para a limpeza: a função `consertar_acento` do kit (pronta) desfaz o erro relendo o texto do jeito certo (`encode("latin-1").decode("utf-8")`). Quando o texto não tem o problema, ela devolve o texto como estava. Ela é chamada dentro da `limpar_texto`, então todo texto passa por ela.